# Cell Reconstruction from Gradient Vectors

Based on the rotated gradient vectors of sym8 + Sobel filtering, this notebook tries to reconstruct cell shape from the vectors and their magnitudes.

## File Import

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    # "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/251205_MonoCa9_Phalloidin_Cellbrite_LowConf/Ca922_Mono_LowConf_Phal_Cellbrite_02_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 4000,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[5], file_pattern[0], verbose=True)

cy5_container = ImageContainer(files[0],config)
dapi_container = ImageContainer(files[1],config)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/cell_reconstruction")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
from image_processing_tools.util.phalloidin_processing import process_actin_fibers
import gc
import matplotlib.pyplot as plt

img = cy5_container.merge()
dp_img = dapi_container.merge()

arrow_params = {
    'flip_opposing_vectors': False,
    'use_log_magnitude': True,
    'clip_magnitude': True,
}

gx, gy = process_actin_fibers(img, edge_method="sobel", **arrow_params)

output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_edge_detection.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

In [ ]:
import matplotlib.pyplot as plt
from image_processing_tools.util.phalloidin_processing import plot_gradient_field_on_empty_axis

# Assuming gx and gy are already defined in your environment
# and plot_gradient_field_on_empty_axis is imported/available

fig, axes = plt.subplots(1,2,figsize=(12, 6))

# Parameters to make arrows smaller and denser
arrow_params = {
    'step': 10,              # Smaller step = more arrows (default was 20 or 50)
    'arrow_scale': 0.8,      # Scale of arrow length relative to the step size
    'arrow_width': 0.001,    # Thinner arrows to avoid cluttering
    'alpha': 0.8,
    'rotate_90': True,       # Standard rotation for this pipeline
    'color_by_magnitude': False,
    'use_log_magnitude': True,
    'colormap': 'viridis',
    'flip_opposing_vectors': False,
    'clip_magnitude': False,
}

plot_gradient_field_on_empty_axis(axes[0], gx, gy, **arrow_params)
axes[0].set_title('Color by Direction')

arrow_params['color_by_magnitude'] = True
plot_gradient_field_on_empty_axis(axes[1], gx, gy, **arrow_params)
axes[1].set_title('Color by Magnitude')

plt.tight_layout()
plt.show()

output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_gradient_only.png"

plt.savefig(output_filename, dpi=450, bbox_inches='tight')
plt.close(fig)
gc.collect()
print(f"Saved: {output_filename}")

In [ ]:
from image_processing_tools.util.phalloidin_processing import plot_gradient_magnitude_histogram

plot_gradient_magnitude_histogram(gx, gy, use_log_magnitude = True, log_frequency = False, clip_magnitude=True)

output_filename = current_output_dir / f"{Path(fitc_container.name).stem}_magnitude_histogram.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

In [ ]:
from image_processing_tools.util.phalloidin_processing import plot_3d_gradient_magnitude_plotly
from pathlib import Path

downsample_ratio = 4
output_filename = current_output_dir / f"{Path(fitc_container.name).stem}_downsample_{downsample_ratio}x_3D_surface.html"

plot_3d_gradient_magnitude_plotly(gx, gy, 
                                  downsample=downsample_ratio, 
                                  flip_opposing_vectors=False, 
                                  smooth_sigma = 10, 
                                  smooth_angle_sigma = 3,
                                  use_log_magnitude=True,
                                  save_path = output_filename,
                                  show_plot = False,
                                  clip_magnitude=True)

In [ ]:
from image_processing_tools.util.phalloidin_processing import fit_gradient_magnitude_gmm

gmm_plot_args = {
    'n_components':2, 
    'use_log_magnitude': True, 
    'clip_magnitude': True,
    'show_plot': True,
    'prob_colormap': 'uncertainty'
}

fit_gradient_magnitude_gmm(gx, gy, **gmm_plot_args)

output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_{gmm_plot_args.get('prob_colormap')}_gmm_fit.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")